# 03 — Bayesian Structural Time Series with Storage Exogenous

## Mathematical Formulation
We specify a Bayesian Structural Time Series model using **NumPyro** and **JAX**:
$$y_t = \tau_t + (\beta_{\text{normal}} \cdot (1 - R_t) + \beta_{\text{crisis}} \cdot R_t) \cdot \text{storage}_t + \gamma \cdot R_t + \epsilon_t$$
where:
- $y_t$ is the log TTF price,
- $\tau_t$ represents the structural equilibrium level and trend,
- $\beta_{\text{normal}}$ and $\beta_{\text{crisis}}$ quantify regime-conditional marginal storage elasticities,
- $R_t$ is the binary market regime indicator,
- $\epsilon_t \sim \mathcal{N}(0, \sigma^2_{\epsilon})$ is the observation innovation.

Estimation is performed via the No-U-Turn Sampler (NUTS) with Hamiltonian Monte Carlo.

In [ ]:
import pandas as pd
import arviz as az
from src.models.bayesian_ts import run_bayesian_model

df_features = pd.read_parquet('data/processed/feature_matrix.parquet')
bsts, idata = run_bayesian_model(df=df_features, dry_run=False)
diagnostics = bsts.verify_convergence()
print('Convergence Verification:', diagnostics)

### 3.1 Posterior Summary & Credible Intervals
Evaluating highest density intervals (95% HDIs) and effective sample sizes across structural parameters:

In [ ]:
ci = bsts.get_credible_intervals(prob=0.95)
for var, (low, med, high) in ci.items():
    print(f'{var:<15}: Median = {med:7.4f}, 95% HDI = [{low:7.4f}, {high:7.4f}]')

display(bsts.summary_df)

### 3.2 Visualizing Marginal Storage Elasticity
Comparing the posterior parameter densities between Normal and Crisis regimes:

In [ ]:
from src.visualization.plots import plot_bayesian_posteriors
plot_bayesian_posteriors(bsts.summary_df)